# Notebook 01 — Dataset Overview and Experiment 1 Baseline

**Research context:** Component 4 — AI-Driven Governance, Compliance and Trust Infrastructure  
**Research question:** What is the composition of the 200-record governance complaint dataset, and how well does a source-group-aware TF-IDF + Logistic Regression baseline classify governance complaint categories?

**Experiment 1** uses `StratifiedGroupKFold` with `Source_Group_ID` as the grouping key. This prevents leakage from records sharing the same audit source, but does not address cross-source text-template similarity (see Notebook 02).

**Caveat:** All evaluation results shown here are **development cross-validation outputs**, not untouched held-out test results. They are exploratory evidence only.

---

## How to use this notebook

**Local Jupyter:** Open with `jupyter notebook` or `jupyter lab` from within the ML directory, or set `ML_ROOT` in the Setup cell below.

**Google Colab:**
1. Download the repository as a ZIP from GitHub (or receive it from your supervisor).
2. Upload the ZIP to Colab via `Files → Upload`.
3. Unzip: `!unzip <your_zip>.zip -d /content/project`
4. Set `ML_ROOT` in the Setup cell to: `/content/project/StateLandGovernance/src/Modules/GovernanceIntelligence/ML`
5. If required packages are missing, run the optional installation cell.

**This notebook does not train models, modify datasets, or overwrite results.**

In [ ]:
# =============================================================================
# SETUP — Configure ML_ROOT to point to the ML module directory
# =============================================================================
import sys
from pathlib import Path

# *** EDIT THIS PATH if running outside the repository ***
# Local default: resolve relative to this notebook's own location
ML_ROOT = Path(globals().get('__vsc_ipynb_file__', __file__) if '__file__' in dir() else '.').resolve().parent
# If the above auto-detection fails, uncomment and set manually:
# ML_ROOT = Path("/path/to/ML")  # e.g. /content/project/.../ML on Colab

SRC_DIR = ML_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"ML_ROOT: {ML_ROOT}")
print(f"ML_ROOT exists: {ML_ROOT.exists()}")

In [ ]:
# =============================================================================
# OPTIONAL — Dependency installation (run only if packages are missing)
# Do NOT run automatically in a shared environment
# =============================================================================
# Uncomment to install:
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(ML_ROOT / "requirements.txt")], check=True)

In [ ]:
# =============================================================================
# IMPORTS AND VERSION DISPLAY
# =============================================================================
import json
import hashlib

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay
)

print(f"Python:      {sys.version.split()[0]}")
print(f"NumPy:       {np.__version__}")
print(f"Pandas:      {pd.__version__}")
print(f"scikit-learn:{sklearn.__version__}")
print(f"Matplotlib:  {matplotlib.__version__}")

In [ ]:
# =============================================================================
# ARTIFACT PATHS AND IDENTITY CHECK
# =============================================================================
DATA_CSV   = ML_ROOT / "data"    / "state_land_governance_confirmed_200.csv"
EXP1_OOF   = ML_ROOT / "results" / "out_of_fold_predictions.csv"
EXP1_METRICS = ML_ROOT / "results" / "baseline_metrics.json"
EXP1_PC_CSV  = ML_ROOT / "results" / "per_class_metrics.csv"
EXP1_CM_CSV  = ML_ROOT / "results" / "confusion_matrix.csv"

# Check all required files exist
required = {"dataset": DATA_CSV, "oof_predictions": EXP1_OOF, "metrics_json": EXP1_METRICS}
for name, path in required.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"  {status}: {name} -> {path.name}")

missing = [p for p in required.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"{len(missing)} required file(s) missing. "
        "Check ML_ROOT and ensure the repository artifacts are present."
    )

# Dataset SHA-256
dataset_hash = hashlib.sha256(DATA_CSV.read_bytes()).hexdigest()
print(f"\nDataset SHA-256: {dataset_hash}")
EXPECTED_HASH = "26979f18f292f8a33a5ff960123a168f0652a5d8e11032b930b4dc7fcf070c61"
print(f"Expected SHA-256: {EXPECTED_HASH}")
print(f"Hash match: {dataset_hash == EXPECTED_HASH}")

## 1. Dataset Overview

In [ ]:
df = pd.read_csv(DATA_CSV)
print(f"Total records: {len(df)}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# Label distribution
label_counts = df["ML_Label_4Class"].value_counts()
print("Label counts:")
print(label_counts.to_string())

fig, ax = plt.subplots(figsize=(9, 4))
label_counts.plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Count")
ax.set_title("Dataset: Record Count per Governance Category (n=200)")
ax.invert_yaxis()
for bar in ax.patches:
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f"{int(bar.get_width())}", va="center")
plt.tight_layout()
plt.show()

In [ ]:
# Data quality: missing values
key_cols = ["Research_ID", "Canonical_English_Text", "ML_Label_4Class", "Source_Group_ID"]
print("Missing values in key columns:")
print(df[key_cols].isnull().sum().to_string())
print(f"\nBlank text fields: {(df['Canonical_English_Text'].str.strip() == '').sum()}")
print(f"Unique Source_Group_IDs: {df['Source_Group_ID'].nunique()}")

In [ ]:
# Dataset provenance: distinguish source types
# Gold_Label_Status column indicates label review status
if "Gold_Label_Status" in df.columns:
    print("Label review status breakdown:")
    print(df["Gold_Label_Status"].value_counts().to_string())
    print(
        "\nNote: 'Proposed research label - final human/domain review required' indicates records "
        "whose labels were assigned by the researcher and are pending domain validation. "
        "'Commissioner-confirmed label' indicates labels confirmed by a domain authority."
    )

if "Narrative_Type" in df.columns:
    print("\nNarrative type breakdown:")
    print(df["Narrative_Type"].value_counts().to_string())
    print(
        "\nNote: 'Direct governance complaint / allegation narrative (source-derived)' records are "
        "summaries derived from audit reports or official source documents, not verbatim complaint text. "
        "'Governance complaint narrative' records are researcher-drafted scenarios. "
        "This distinction affects how text variation should be interpreted."
    )

## 2. Experiment 1 — Source-Group-Aware Baseline

**Evaluation method:** `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)`  
**Groups:** `Source_Group_ID` — prevents records from the same audit report appearing in both train and test in the same fold.  
**Pipeline:** TF-IDF (unigrams + bigrams, sublinear TF, English stop words) → Logistic Regression (balanced class weight, L2, C=1.0).  
**These results are computed from the saved OOF prediction CSV, not by retraining.**

In [ ]:
oof = pd.read_csv(EXP1_OOF)
print(f"OOF predictions loaded: {len(oof)} rows, {oof['Research_ID'].nunique()} unique IDs")

LABEL_ORDER = [
    "Administrative / Procedural / Integrity",
    "Lease Revenue / Payment / Enforcement",
    "Unauthorized Allocation / Transfer / Use",
    "Protected / Environmental Lease Misuse",
]

y_true = oof["True_Label"].tolist()
y_pred = oof["Predicted_Label"].tolist()

In [ ]:
# Overall metrics
acc      = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro",    labels=LABEL_ORDER, zero_division=0)
wt_f1    = f1_score(y_true, y_pred, average="weighted", labels=LABEL_ORDER, zero_division=0)
macro_p  = precision_score(y_true, y_pred, average="macro", labels=LABEL_ORDER, zero_division=0)
macro_r  = recall_score(y_true, y_pred, average="macro",    labels=LABEL_ORDER, zero_division=0)
n_wrong  = int(sum(a != b for a, b in zip(y_true, y_pred)))

print("Experiment 1 — Overall OOF Metrics (source-grouped CV, n=200)")
print(f"  Accuracy:           {acc:.4f}")
print(f"  Macro-Precision:    {macro_p:.4f}")
print(f"  Macro-Recall:       {macro_r:.4f}")
print(f"  Macro-F1:           {macro_f1:.4f}")
print(f"  Weighted-F1:        {wt_f1:.4f}")
print(f"  Misclassified:      {n_wrong} / {len(y_true)}")

In [ ]:
# Per-class metrics
cm = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)
pc_rows = []
for i, lbl in enumerate(LABEL_ORDER):
    tp = cm[i,i]; fp = int(cm[:,i].sum() - tp); fn = int(cm[i,:].sum() - tp)
    tn = int(cm.sum() - tp - fp - fn)
    p  = tp/(tp+fp) if tp+fp>0 else 0
    r  = tp/(tp+fn) if tp+fn>0 else 0
    f1 = 2*tp/(2*tp+fp+fn) if (2*tp+fp+fn)>0 else 0
    fpr= fp/(fp+tn) if (fp+tn)>0 else 0
    pc_rows.append({"Class": lbl[:35], "Precision": round(p,4), "Recall": round(r,4),
                    "F1": round(f1,4), "FPR": round(fpr,4), "Support": tp+fn})
pc_df = pd.DataFrame(pc_rows).set_index("Class")
print("Per-class metrics (Experiment 1):")
print(pc_df.to_string())
print("\nNote: F1 values are per-class F1 scores, not macro-averaged.")

In [ ]:
# Confusion matrix
short_labels = [l.split("/")[0].strip()[:20] for l in LABEL_ORDER]
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=short_labels)
fig, ax = plt.subplots(figsize=(8, 7))
disp.plot(ax=ax, colorbar=True, cmap="Blues")
ax.set_title("Experiment 1 — Confusion Matrix\n(Source-Grouped CV, OOF Predictions, n=200)")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
plt.tight_layout()
plt.show()

In [ ]:
# Fold breakdown
if "Fold" in oof.columns:
    fold_summary = oof.groupby("Fold").apply(
        lambda g: pd.Series({
            "n": len(g),
            "correct": (g["True_Label"] == g["Predicted_Label"]).sum(),
            "accuracy": round((g["True_Label"] == g["Predicted_Label"]).mean(), 4)
        })
    )
    print("Per-fold accuracy:")
    print(fold_summary.to_string())

## 3. Interpretation and Limitations

- Experiment 1 groups records by audit source (`Source_Group_ID`) to reduce leakage. However, records from different sources may still share identical or near-identical narrative templates, which could inflate cross-validation scores. See Notebook 02 for the template-aware analysis.
- The four governance categories have substantial boundary overlap, especially between *Revenue* and *Administrative* violations, and between *Unauthorized Allocation* and *Environmental Misuse*. High proportions of errors occur at these boundaries.
- All labels are *research-assigned* and pending domain validation, unless explicitly marked as Commissioner-confirmed.
- These development CV results should not be interpreted as model performance on genuinely unseen data.

### Optional: Reproduce Experiment 1

To reproduce from scratch (requires full environment and dataset):
```bash
cd <ML_ROOT>
python src/evaluate_experiment1.py
```
This will overwrite results in `results/`. Do not run if you want to preserve existing artifacts.